In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
import warnings

# --- 0. 环境设置 ---
warnings.filterwarnings('ignore')
sns.set(style="whitegrid", font_scale=1.1)
# 设置中文字体（如果需要）
# plt.rcParams['font.sans-serif'] = ['SimHei'] 
# plt.rcParams['axes.unicode_minus'] = False 

print("="*50)
print(" 1. 开始加载数据")
print("="*50)

# --- 1. 数据加载 ---
try:
    df_train = pd.read_csv('dataset/train.csv')
    df_test = pd.read_csv('dataset/test.csv')
    df_original = pd.read_csv('dataset/final_depression_dataset_1.csv')
    print("train.csv, test.csv, final_depression_dataset_1.csv 加载完毕。")
except FileNotFoundError as e:
    print(f"错误：未能找到文件。请确保所有 CSV 文件都在。")
    print(f"具体错误: {e}")
    # exit() # 在真实脚本中会退出

# --- 2. [新增] 探索性数据分析 (EDA) 可视化 ---
print("\n" + "="*50)
print(" 2. 开始生成 EDA 可视化图表")
print("="*50)

# 可视化 2.1: 目标变量分布
print("  正在生成: 1_target_distribution.png")
plt.figure(figsize=(8, 6))
ax = sns.countplot(data=df_train, x='Depression', palette=['#4374B3', '#FF6347'])
plt.title('Target Variable (Depression) Distribution', fontsize=16)
plt.xlabel('Depression Status', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks([0, 1], ['Not Depressed (0)', 'Depressed (1)'])
total = len(df_train)
for p in ax.patches:
    height = p.get_height()
    ax.text(p.get_x() + p.get_width() / 2., height + 3, f'{height/total:.1%}', ha="center", fontsize=12)
plt.savefig('1_target_distribution.png', dpi=150)
plt.close() # 关闭图像，防止在 notebook 中显示两次

# 可视化 2.2: 缺失值热力图
print("  正在生成: 2_missing_values_heatmap.png")
plt.figure(figsize=(18, 8))
plt.subplot(1, 2, 1)
features_train = df_train.drop(['id', 'Name', 'Depression'], axis=1)
sns.heatmap(features_train.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Values in Train Data', fontsize=14)
plt.subplot(1, 2, 2)
features_test = df_test.drop(['id', 'Name'], axis=1)
sns.heatmap(features_test.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Values in Test Data', fontsize=14)
plt.tight_layout()
plt.savefig('2_missing_values_heatmap.png', dpi=150)
plt.close()

# 可视化 2.3: 分类特征 vs 目标
print("  正在生成: 3_categorical_vs_target.png")
plot_cat_cols = [
    'Gender', 'Working Professional or Student', 'Sleep Duration',
    'Dietary Habits', 'Have you ever had suicidal thoughts ?', 
    'Family History of Mental Illness'
]
fig, axes = plt.subplots(2, 3, figsize=(20, 14))
axes = axes.flatten()
for i, col in enumerate(plot_cat_cols):
    if col in df_train.columns:
        sns.countplot(data=df_train, x=col, hue='Depression', ax=axes[i], palette='pastel')
        axes[i].set_title(f'"{col}" vs Depression', fontsize=14)
        axes[i].set_xlabel(None)
        axes[i].set_ylabel('Count')
        axes[i].tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig('3_categorical_vs_target.png', dpi=150)
plt.close()

# 可视化 2.4: 数值特征 vs 目标
print("  正在生成: 4_numerical_vs_target.png")
num_cols = df_train.select_dtypes(include=np.number).columns.drop(['id', 'Depression'])
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.kdeplot(data=df_train, x=col, hue='Depression', ax=axes[i], 
                fill=True, common_norm=False, palette=['#4374B3', '#FF6347'])
    axes[i].set_title(f'Distribution of "{col}"', fontsize=14)
    axes[i].set_xlabel(col, fontsize=12)
    axes[i].set_ylabel('Density')
for j in range(i + 1, len(axes)):
    axes[j].axis('off')
plt.tight_layout()
plt.savefig('4_numerical_vs_target.png', dpi=150)
plt.close()
print("EDA 可视化图表生成完毕。")


# --- 3. 数据准备 (用于建模) ---
print("\n" + "="*50)
print(" 3. 开始准备建模数据")
print("="*50)

# 存储测试集的 ID
test_ids = df_test['id']

# 转换外部数据的目标变量
df_original['Depression'] = df_original['Depression'].map({'Yes': 1, 'No': 0})

# 定义特征 (X) 和目标 (y)
y = df_train['Depression']
X = df_train.drop(['id', 'Name', 'Depression'], axis=1)

X_test = df_test.drop(['id', 'Name'], axis=1)

y_original = df_original['Depression']
X_original = df_original.drop(['Name', 'Depression'], axis=1)

# 确保列顺序一致
X_test = X_test[X.columns]
X_original = X_original[X.columns]
print("建模数据准备完毕。")

# --- 4. 预处理管道 ---
print("\n" + "="*50)
print(" 4. 开始构建预处理管道")
print("="*50)

numerical_cols = X.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

# 数值特征处理：均值插补 + 标准化
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# 分类特征处理：众数插补 + 独热编码 (One-Hot)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# 使用 ColumnTransformer 组合
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='passthrough'
)
print("预处理管道构建完毕。")

# --- 5. 模型定义与超参数 ---
print("\n" + "="*50)
print(" 5. 开始定义模型")
print("="*50)

# (A) 逻辑回归 (Logistic Regression)
lr_hyperparameters = {
    'C': 1.0,
    'penalty': 'l2',
    'solver': 'lbfgs',
    'max_iter': 1000,
    'random_state': 42
}
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(**lr_hyperparameters))
])
print("定义: LogisticRegression")

# (B) 线性 SVM (LinearSVC) + 概率校准
svm_hyperparameters = {
    'C': 1.0,
    'penalty': 'l2',
    'loss': 'squared_hinge',
    'max_iter': 5000,
    'dual': 'auto',
    'random_state': 42
}
svm_base_model = LinearSVC(**svm_hyperparameters)
svm_calibrated_model = CalibratedClassifierCV(svm_base_model, cv=3)

svm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', svm_calibrated_model)
])
print("定义: LinearSVC (with CalibratedClassifierCV)")

# --- 6. 交叉验证训练 ---
print("\n" + "="*50)
print(" 6. 开始交叉验证训练")
print("="*50)

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# 用于存储 OOF 预测（用于混淆矩阵）
oof_preds_lr_binary = np.zeros(len(X))
oof_preds_svm_binary = np.zeros(len(X))

def run_training(model_pipeline, X, y, X_test, X_original, y_original):
    model_name = model_pipeline.steps[-1][1].__class__.__name__
    # 修复 CalibratedClassifierCV 的命名
    if "CalibratedClassifierCV" in model_name:
        model_name = f"Calibrated({model_pipeline.steps[-1][1].estimator.__class__.__name__})"

    print(f"\n--- [开始] 训练模型: {model_name} ---")
    
    oof_preds_proba = np.zeros(len(X))
    test_preds_proba = np.zeros(len(X_test))
    val_scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"  -- Fold {fold+1}/{N_FOLDS} --")
        
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
        
        X_train_aug = pd.concat([X_train_fold, X_original], ignore_index=True)
        y_train_aug = pd.concat([y_train_fold, y_original], ignore_index=True)
        
        model = clone(model_pipeline)
        model.fit(X_train_aug, y_train_aug)
        
        y_val_pred_proba = model.predict_proba(X_val_fold)[:, 1]
        oof_preds_proba[val_idx] = y_val_pred_proba
        
        y_val_pred_binary = (y_val_pred_proba > 0.5).astype(int)
        score = accuracy_score(y_val_fold, y_val_pred_binary)
        val_scores.append(score)
        print(f"    Fold {fold+1} Accuracy: {score:.6f}")
        
        test_preds_proba += model.predict_proba(X_test)[:, 1] / skf.n_splits

    overall_oof_accuracy = accuracy_score(y, (oof_preds_proba > 0.5).astype(int))
    mean_val_accuracy = np.mean(val_scores)
    
    print(f"\n  模型训练完毕。")
    print(f"  平均 5-Fold 验证准确率: {mean_val_accuracy:.6f}")
    print(f"  总体 OOF 验证准确率:   {overall_oof_accuracy:.6f}")
    print(f"--- [结束] ---")
    
    # 返回 OOF 预测（用于混淆矩阵）和测试集预测
    return test_preds_proba, overall_oof_accuracy, (oof_preds_proba > 0.5).astype(int)

# 执行训练
test_preds_lr, acc_lr, oof_preds_lr_binary = run_training(lr_pipeline, X, y, X_test, X_original, y_original)
test_preds_svm, acc_svm, oof_preds_svm_binary = run_training(svm_pipeline, X, y, X_test, X_original, y_original)

# --- 7. [新增] 模型评估可视化 ---
print("\n" + "="*50)
print(" 7. 开始生成模型评估可视化")
print("="*50)

# 选择最佳模型用于可视化
if acc_lr > acc_svm:
    print("选择 LogisticRegression 作为最佳模型进行评估。")
    best_model_name = 'LogisticRegression'
    best_oof_preds = oof_preds_lr_binary
    best_test_preds = test_preds_lr
else:
    print("选择 Calibrated(LinearSVC) 作为最佳模型进行评估。")
    best_model_name = 'Calibrated(LinearSVC)'
    best_oof_preds = oof_preds_svm_binary
    best_test_preds = test_preds_svm

# 可视化 7.1: OOF 混淆矩阵
print("  正在生成: 5_oof_confusion_matrix.png")
cm = confusion_matrix(y, best_oof_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot(cmap=plt.cm.Blues)
plt.title(f'OOF Confusion Matrix for {best_model_name}')
plt.savefig('5_oof_confusion_matrix.png', dpi=150)
plt.close()

# 可视化 7.2: 逻辑回归特征重要性
print("  正在生成: 6_lr_feature_importance.png")
# 为了获得稳定的特征重要性，我们在所有数据上重新训练一次 LR 模型
print("  (正在所有训练数据上重新训练 LR 模型以提取特征...)\n")
X_aug_all = pd.concat([X, X_original], ignore_index=True)
y_aug_all = pd.concat([y, y_original], ignore_index=True)

final_lr_pipeline = clone(lr_pipeline)
final_lr_pipeline.fit(X_aug_all, y_aug_all)

# 提取特征名称和系数
try:
    preprocessor_fitted = final_lr_pipeline.named_steps['preprocessor']
    model_fitted = final_lr_pipeline.named_steps['classifier']
    
    # 获取数值特征名称
    numeric_features = numerical_cols
    
    # 获取 OneHot 编码后的分类特征名称
    categorical_features = preprocessor_fitted.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_cols)
    
    # 合并所有特征名称
    all_features = list(numeric_features) + list(categorical_features)
    
    # 获取系数
    coefficients = model_fitted.coef_[0]
    
    # 创建 DataFrame
    coef_df = pd.DataFrame({'Feature': all_features, 'Coefficient': coefficients})
    coef_df = coef_df.sort_values(by='Coefficient', ascending=False)

    # 绘制 Top 20 和 Bottom 20
    top_n = 20
    plot_df = pd.concat([coef_df.head(top_n), coef_df.tail(top_n)])
    
    plt.figure(figsize=(12, 16))
    sns.barplot(x='Coefficient', y='Feature', data=plot_df, palette='vlag')
    plt.title(f'Logistic Regression Feature Importance (Top/Bottom {top_n})', fontsize=16)
    plt.xlabel('Coefficient Value', fontsize=12)
    plt.ylabel('Feature', fontsize=12)
    plt.tight_layout()
    plt.savefig('6_lr_feature_importance.png', dpi=150)
    plt.close()

except Exception as e:
    print(f"  生成特征重要性图时出错: {e}")
    print("  (这可能是由于管道结构变化引起的，跳过此图。)")

print("模型评估可视化生成完毕。")

# --- 8. 生成提交文件 ---
print("\n" + "="*50)
print(" 8. 开始生成提交文件")
print("="*50)

final_test_labels = (best_test_preds > 0.5).astype(int)
submission_df = pd.DataFrame({'id': test_ids, 'Depression': final_test_labels})
submission_filename = f'submission_{best_model_name.lower().replace("(", "_").replace(")", "")}.csv'
submission_df.to_csv(submission_filename, index=False)

print(f"提交文件已保存为: {submission_filename}")
print(submission_df.head())
print("\n" + "="*50)
print(" 脚本执行完毕 ")
print("="*50)

 1. 开始加载数据
train.csv, test.csv, final_depression_dataset_1.csv 加载完毕。

 2. 开始生成 EDA 可视化图表
  正在生成: 1_target_distribution.png
train.csv, test.csv, final_depression_dataset_1.csv 加载完毕。

 2. 开始生成 EDA 可视化图表
  正在生成: 1_target_distribution.png
  正在生成: 2_missing_values_heatmap.png
  正在生成: 2_missing_values_heatmap.png
  正在生成: 3_categorical_vs_target.png
  正在生成: 3_categorical_vs_target.png
  正在生成: 4_numerical_vs_target.png
  正在生成: 4_numerical_vs_target.png
EDA 可视化图表生成完毕。

 3. 开始准备建模数据
建模数据准备完毕。

 4. 开始构建预处理管道
预处理管道构建完毕。

 5. 开始定义模型
定义: LogisticRegression
定义: LinearSVC (with CalibratedClassifierCV)

 6. 开始交叉验证训练

--- [开始] 训练模型: LogisticRegression ---
  -- Fold 1/5 --
EDA 可视化图表生成完毕。

 3. 开始准备建模数据
建模数据准备完毕。

 4. 开始构建预处理管道
预处理管道构建完毕。

 5. 开始定义模型
定义: LogisticRegression
定义: LinearSVC (with CalibratedClassifierCV)

 6. 开始交叉验证训练

--- [开始] 训练模型: LogisticRegression ---
  -- Fold 1/5 --
    Fold 1 Accuracy: 0.938131
    Fold 1 Accuracy: 0.938131
  -- Fold 2/5 --
  -- Fold 2/5 --
    Fold 2 Accuracy: 0.937171